In [1]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound

AWS_PROFILE = "training"
AWS_REGION = "us-east-1"

try:
    session = boto3.Session(
        profile_name=AWS_PROFILE,
        region_name=AWS_REGION
    )

    print("Region:", session.region_name)

    sts = session.client("sts")

    identity = sts.get_caller_identity()

    print("Account   :", identity["Account"])
    print("User ID   :", identity["UserId"])
    print("Principal :", identity["Arn"])

except ProfileNotFound as e:
    print("Profile problem:", e)

except NoCredentialsError:
    print("AWS credentials not found")

except ClientError as e:
    print("AWS error:", e)

Region: us-east-1
Account   : 608553548146
User ID   : AIDAY3ME6HFZKAX5ZDECC
Principal : arn:aws:iam::608553548146:user/cloud_user


In [2]:
s3 = session.client("s3")

try:
    response = s3.list_buckets()

    print("Available S3 buckets:")
    for bucket in response.get("Buckets", []):
        print(f"- {bucket['Name']}")

except NoCredentialsError:
    print("AWS credentials were not found.")
except ClientError as exc:
    print(f"AWS API error: {exc}")
except BotoCoreError as exc:
    print(f"AWS SDK error: {exc}")

Available S3 buckets:
- aws-glue-assets-608553548146-us-east-1
- gks-datalake2


In [ ]:
import boto3
import random
import uuid
import datetime
import json
import time

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

KINESIS_STREAM_NAME = "gks-kinesis-stream"
AWS_REGION = "us-east-1"

SAMPLES = 10000
DELAY = 5  # seconds between invoices

countries = [
    "USA", "CA", "IN", "AT", "BE", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT",
    "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"
]

stock_codes = [
    "85123A",
    "71053",
    "84406B",
    "84406G",
    "84406E"
]

customer_codes = [
    17850,
    13047,
    12583,
    17850
]


# ---------------------------------------------------------
# Kinesis Client
# ---------------------------------------------------------

kinesis_client = session.client("kinesis")


# ---------------------------------------------------------
# Generate invoices
# ---------------------------------------------------------

for i in range(SAMPLES):

    # Generate invoice number
    invoice_no = uuid.uuid4().hex[:8].upper()

    customer_code = random.choice(customer_codes)
    country = random.choice(countries)

    # UTC timestamp in ISO-8601 format
    invoice_date = datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat()

    # Each invoice contains 3-10 items
    number_of_items = random.randint(3, 10)

    print(
        f"\nInvoice {i + 1}/{SAMPLES}: "
        f"{invoice_no} | "
        f"Customer={customer_code} | "
        f"Country={country} | "
        f"Items={number_of_items}"
    )

    for j in range(number_of_items):

        quantity = random.randint(1, 10)
        unit_price = float(random.randint(1, 5))
        stock_code = random.choice(stock_codes)

        invoice = {
            "InvoiceNo": invoice_no,
            "StockCode": stock_code,
            "Quantity": quantity,
            "Description": "TODO",
            "InvoiceDate": invoice_date,
            "UnitPrice": unit_price,
            "CustomerID": customer_code,
            "Country": country
        }

        # Convert Python dictionary to JSON
        invoice_str = json.dumps(invoice)

        print("Sending:", invoice_str)

        # Send one invoice-line event to Kinesis
        response = kinesis_client.put_record(
            StreamName=KINESIS_STREAM_NAME,
            Data=invoice_str.encode("utf-8"),
            PartitionKey=country
        )

        print(
            "Kinesis:",
            "ShardId =", response["ShardId"],
            "SequenceNumber =", response["SequenceNumber"]
        )

    # Wait before generating the next invoice
    time.sleep(DELAY)


Invoice 1/10000: AA855998 | Customer=12583 | Country=ES | Items=5
Sending: {"InvoiceNo": "AA855998", "StockCode": "84406B", "Quantity": 7, "Description": "TODO", "InvoiceDate": "2026-09-21T11:03:07.657892+00:00", "UnitPrice": 3.0, "CustomerID": 12583, "Country": "ES"}
Kinesis: ShardId = shardId-000000000000 SequenceNumber = 49678545858210232253980413603675935705142569586921570306
Sending: {"InvoiceNo": "AA855998", "StockCode": "84406B", "Quantity": 9, "Description": "TODO", "InvoiceDate": "2026-09-21T11:03:07.657892+00:00", "UnitPrice": 2.0, "CustomerID": 12583, "Country": "ES"}
Kinesis: ShardId = shardId-000000000000 SequenceNumber = 49678545858210232253980413603684398185879871991144513538
Sending: {"InvoiceNo": "AA855998", "StockCode": "85123A", "Quantity": 7, "Description": "TODO", "InvoiceDate": "2026-09-21T11:03:07.657892+00:00", "UnitPrice": 3.0, "CustomerID": 12583, "Country": "ES"}
Kinesis: ShardId = shardId-000000000000 SequenceNumber = 496785458582102322539804136036952785182